# Dissertation Final Figures

This notebook creates the final dissertation figures from saved results.

It does **not** rerun classification, LOSO, EEGNet training, ICA, ERP extraction, or subject-level feature extraction.

It reads saved result tables and saved result artifacts, then exports thesis-ready figures under:

`results/thesis_artifacts/dissertation_final_figures/`

Final outputs:

- `fig05_01_within_subject_auc.pdf`
- `fig05_02_loso_auc.pdf`
- `fig06_01_erp_difference_waveforms.pdf`
- `fig06_02_time_resolved_decoding.pdf`
- `fig07_01_subject_vs_classifier_variability.pdf`
- `fig07_02_subject_quality_vs_auc.pdf`
- `fig07_03_incremental_r2.pdf`

The TikZ conceptual figures for Chapters 1, 3, and 4 are written directly in the LaTeX thesis file and are not generated here.

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from pathlib import Path
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------


RESULTS_DIR = PROJECT_ROOT / "results"

ARTIFACT_DIR = RESULTS_DIR / "thesis_artifacts" / "dissertation_final_figures"
FIGURES_DIR = ARTIFACT_DIR / "figures"
TABLES_DIR = ARTIFACT_DIR / "tables"
SOURCE_COPY_DIR = ARTIFACT_DIR / "source_artifacts"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_COPY_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Results dir  :", RESULTS_DIR)
print("Artifact dir :", ARTIFACT_DIR)
print("Figures dir  :", FIGURES_DIR)
print("Tables dir   :", TABLES_DIR)
print("Source dir   :", SOURCE_COPY_DIR)

In [ ]:
# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

# ------------------------------------------------------------
# Plot settings: thesis-ready white background
# ------------------------------------------------------------

plt.style.use("default")

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,

    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",

    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    "axes.grid": False,
})

In [ ]:
# ------------------------------------------------------------
# General helper functions
# ------------------------------------------------------------

def is_inside_artifact_dir(path):
    """
    Return True if path is inside the final dissertation artifact directory.
    """
    path = Path(path).resolve()
    artifact_root = ARTIFACT_DIR.resolve()

    try:
        path.relative_to(artifact_root)
        return True
    except ValueError:
        return False


def find_all(patterns, root=PROJECT_ROOT, exclude_final_artifacts=True):
    """
    Return all matching files from one or more glob patterns.
    Excludes the final dissertation artifact folder by default so that
    copied outputs are not re-used as source files.
    """
    if isinstance(patterns, str):
        patterns = [patterns]

    matches = []
    for pattern in patterns:
        matches.extend(sorted(root.glob(pattern)))

    if exclude_final_artifacts:
        matches = [p for p in matches if not is_inside_artifact_dir(p)]

    return matches


def find_first(patterns, root=PROJECT_ROOT, exclude_final_artifacts=True):
    """
    Return the first matching file from one or more glob patterns.
    Excludes the final dissertation artifact folder by default.
    """
    matches = find_all(
        patterns,
        root=root,
        exclude_final_artifacts=exclude_final_artifacts,
    )

    return matches[0] if matches else None


def require_file(patterns, description, root=PROJECT_ROOT):
    """
    Find a required file. Raise a clear error if absent.
    """
    path = find_first(patterns, root=root)
    if path is None:
        raise FileNotFoundError(
            f"Could not find required file for: {description}\n"
            "Patterns tried:\n"
            + "\n".join(f"  - {p}" for p in patterns)
        )

    print(f"{description}: {path}")
    return path


def require_columns(df, required_cols, table_name):
    """
    Ensure required columns exist.
    """
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"{table_name} is missing required columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )


def clean_dataset_label(value):
    """
    Standardize dataset labels for plots.
    """
    value = str(value)

    replacements = {
        "ds003061": "ds003061",
        "ds005863": "ds005863",
        "ds006466": "ds006466",
        "ds006480": "ds006480 pooled",
        "ds006480_pooled": "ds006480 pooled",
        "ds006480 pooled": "ds006480 pooled",
    }

    return replacements.get(value, value)


def clean_model_label(value):
    """
    Standardize model names for plots.
    """
    value = str(value)

    replacements = {
        "lda": "LDA",
        "LDA": "LDA",
        "logreg": "Logistic Regression",
        "LogReg": "Logistic Regression",
        "logistic_regression": "Logistic Regression",
        "LogisticRegression": "Logistic Regression",
        "Logistic Regression": "Logistic Regression",
        "eegnet": "EEGNet",
        "EEGNet": "EEGNet",
    }

    return replacements.get(value, value)


def normalize_evaluation_setting(value):
    """
    Standardize evaluation-setting labels.
    """
    value = str(value).strip().lower()

    replacements = {
        "within_subject": "Within-subject",
        "within-subject": "Within-subject",
        "within subject": "Within-subject",
        "grouped_within_subject": "Within-subject",
        "grouped within-subject": "Within-subject",
        "loso": "LOSO",
        "leave_one_subject_out": "LOSO",
        "leave-one-subject-out": "LOSO",
        "leave one subject out": "LOSO",
    }

    return replacements.get(value, value)


def save_current_figure(path, bbox_inches="tight"):
    """
    Save current matplotlib figure as PDF and PNG.
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    plt.savefig(path, bbox_inches=bbox_inches)

    if path.suffix.lower() == ".pdf":
        png_path = path.with_suffix(".png")
        plt.savefig(png_path, bbox_inches=bbox_inches)
        print("Saved:", path)
        print("Saved:", png_path)
    else:
        print("Saved:", path)


def prepare_erp_difference_for_plot(df, value_col="difference_uv"):
    """
    Prepare ERP difference values for plotting.

    Some upstream EEG/ERP pipelines store amplitudes in volts even when
    the column name says '_uv'. This helper detects that case and converts
    values to microvolts for thesis plotting.

    Returns:
        df_plot: dataframe with a new column 'difference_plot'
        y_label: y-axis label
        scale_note: text description of what happened
    """
    df_plot = df.copy()

    if value_col not in df_plot.columns:
        raise ValueError(
            f"Column '{value_col}' not found. Available columns: {list(df_plot.columns)}"
        )

    max_abs = df_plot[value_col].abs().max()

    if pd.isna(max_abs):
        raise ValueError("ERP difference column contains only NaN values.")

    # Typical EEG in volts is around 1e-7 to 1e-5.
    # Typical EEG in microvolts is around 0.1 to 20.
    if max_abs < 1e-3:
        df_plot["difference_plot"] = df_plot[value_col] * 1e6
        y_label = "Target - standard (µV)"
        scale_note = "Converted ERP difference values from volts to microvolts."
    else:
        df_plot["difference_plot"] = df_plot[value_col]
        y_label = "Target - standard (µV)"
        scale_note = "ERP difference values already appear to be in microvolts."

    print(scale_note)
    print("Original max abs value:", max_abs)
    print("Plot max abs value    :", df_plot["difference_plot"].abs().max())

    return df_plot, y_label, scale_note


def copy_source_artifact(src_path):
    """
    Copy source artifact into the final artifact folder for traceability.

    If the source file is already inside SOURCE_COPY_DIR, do nothing.
    This prevents SameFileError when rerunning the notebook.
    """
    src_path = Path(src_path).resolve()

    if not src_path.exists():
        return None

    dst_path = (SOURCE_COPY_DIR / src_path.name).resolve()

    # Avoid copying a file onto itself
    if src_path == dst_path:
        print("Source artifact already copied:", src_path)
        return dst_path

    # Avoid unnecessary overwrite if identical-name file already exists
    if dst_path.exists():
        print("Source artifact already exists:", dst_path)
        return dst_path

    shutil.copy2(src_path, dst_path)
    print("Copied source artifact:", dst_path)
    return dst_path

In [ ]:
# ------------------------------------------------------------
# Plotting helper functions
# ------------------------------------------------------------

def grouped_barplot(
        df,
        x_col,
        y_col,
        hue_col,
        output_path,
        ylabel="ROC-AUC",
        xlabel="Dataset",
        ylim=None,
        legend_title=None,
        width_inches=10,
        height_inches=5.5,
):
    """
    Create grouped bar plot using matplotlib only.
    """
    df = df.copy()

    x_order = list(dict.fromkeys(df[x_col].tolist()))
    hue_order = list(dict.fromkeys(df[hue_col].tolist()))

    x = np.arange(len(x_order))
    width = 0.8 / max(len(hue_order), 1)

    fig, ax = plt.subplots(figsize=(width_inches, height_inches))

    for i, hue in enumerate(hue_order):
        sub = df[df[hue_col] == hue].set_index(x_col).reindex(x_order)
        y = sub[y_col].values

        offset = (i - (len(hue_order) - 1) / 2) * width
        ax.bar(x + offset, y, width=width, label=hue)

    ax.set_xticks(x)
    ax.set_xticklabels(x_order, rotation=20, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.grid(axis="y", alpha=0.25)
    ax.legend(title=legend_title, frameon=False)
    fig.tight_layout()

    save_current_figure(output_path)
    plt.show()


def convert_image_to_pdf(source_path, output_path, width=10, height=6):
    """
    Convert an existing saved result image into a thesis-ready PDF.
    Used only when the underlying curve table is unavailable.
    """
    source_path = Path(source_path)
    output_path = Path(output_path)

    img = Image.open(source_path).convert("RGB")

    fig, ax = plt.subplots(figsize=(width, height))
    ax.imshow(img)
    ax.axis("off")

    plt.savefig(output_path, bbox_inches="tight", pad_inches=0.02)
    plt.savefig(output_path.with_suffix(".png"), bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)

    copy_source_artifact(source_path)

    print("Converted source image:", source_path)
    print("Saved:", output_path)
    print("Saved:", output_path.with_suffix(".png"))

In [ ]:
# ------------------------------------------------------------
# Locate final benchmark result tables
# ------------------------------------------------------------

final_summary_path = find_first([
    "results/**/final_max30_wide_timebin_within_loso_model_summary.csv",
    "results/**/final_*within_loso*_model_summary.csv",
])

within_summary_path = find_first([
    "results/**/selected_max30_wide_timebin_model_summary.csv",
    "results/**/selected_*wide_timebin*_model_summary.csv",
])

loso_summary_path = find_first([
    "results/**/loso_model_summary_from_subject_auc.csv",
    "results/**/loso*model*summary*.csv",
])

print("Final within+LOSO summary:", final_summary_path)
print("Within summary           :", within_summary_path)
print("LOSO summary             :", loso_summary_path)

In [ ]:
# ------------------------------------------------------------
# Build within-subject and LOSO plotting tables from saved data
# ------------------------------------------------------------

if final_summary_path is not None:
    benchmark_df = pd.read_csv(final_summary_path)

    dataset_col = "dataset_label" if "dataset_label" in benchmark_df.columns else "dataset"

    require_columns(
        benchmark_df,
        [dataset_col, "model", "within_subject_mean_auc", "loso_mean_auc"],
        "final benchmark summary",
    )

    benchmark_df["dataset_label"] = benchmark_df[dataset_col].apply(clean_dataset_label)
    benchmark_df["model"] = benchmark_df["model"].apply(clean_model_label)

    within_plot_df = benchmark_df[
        ["dataset_label", "model", "within_subject_mean_auc"]
    ].dropna().copy()

    loso_plot_df = benchmark_df[
        ["dataset_label", "model", "loso_mean_auc"]
    ].dropna().copy()

else:
    within_summary_path = require_file(
        [
            "results/**/selected_max30_wide_timebin_model_summary.csv",
            "results/**/selected_*wide_timebin*_model_summary.csv",
        ],
        "within-subject model summary",
    )

    loso_summary_path = require_file(
        [
            "results/**/loso_model_summary_from_subject_auc.csv",
            "results/**/loso*model*summary*.csv",
        ],
        "LOSO model summary",
    )

    within_df = pd.read_csv(within_summary_path)
    loso_df = pd.read_csv(loso_summary_path)

    within_dataset_col = "dataset_label" if "dataset_label" in within_df.columns else "dataset"
    loso_dataset_col = "dataset_label" if "dataset_label" in loso_df.columns else "dataset"

    require_columns(
        within_df,
        [within_dataset_col, "model", "within_subject_mean_auc"],
        "within-subject model summary",
    )

    require_columns(
        loso_df,
        [loso_dataset_col, "model", "loso_mean_auc"],
        "LOSO model summary",
    )

    within_df["dataset_label"] = within_df[within_dataset_col].apply(clean_dataset_label)
    within_df["model"] = within_df["model"].apply(clean_model_label)

    loso_df["dataset_label"] = loso_df[loso_dataset_col].apply(clean_dataset_label)
    loso_df["model"] = loso_df["model"].apply(clean_model_label)

    within_plot_df = within_df[
        ["dataset_label", "model", "within_subject_mean_auc"]
    ].dropna().copy()

    loso_plot_df = loso_df[
        ["dataset_label", "model", "loso_mean_auc"]
    ].dropna().copy()

In [ ]:
# ------------------------------------------------------------
# Standardize order and save plot data
# ------------------------------------------------------------

dataset_order = ["ds003061", "ds005863", "ds006466", "ds006480 pooled"]
model_order = ["LDA", "Logistic Regression", "EEGNet"]

within_plot_df["dataset_label"] = pd.Categorical(
    within_plot_df["dataset_label"],
    categories=dataset_order,
    ordered=True,
)
within_plot_df["model"] = pd.Categorical(
    within_plot_df["model"],
    categories=model_order,
    ordered=True,
)

loso_plot_df["dataset_label"] = pd.Categorical(
    loso_plot_df["dataset_label"],
    categories=dataset_order,
    ordered=True,
)
loso_plot_df["model"] = pd.Categorical(
    loso_plot_df["model"],
    categories=model_order,
    ordered=True,
)

within_plot_df = within_plot_df.sort_values(["dataset_label", "model"]).reset_index(drop=True)
loso_plot_df = loso_plot_df.sort_values(["dataset_label", "model"]).reset_index(drop=True)

within_plot_df.to_csv(TABLES_DIR / "fig05_01_within_subject_auc_plot_data.csv", index=False)
loso_plot_df.to_csv(TABLES_DIR / "fig05_02_loso_auc_plot_data.csv", index=False)

display(within_plot_df)
display(loso_plot_df)

print("Saved:", TABLES_DIR / "fig05_01_within_subject_auc_plot_data.csv")
print("Saved:", TABLES_DIR / "fig05_02_loso_auc_plot_data.csv")

In [ ]:
# ------------------------------------------------------------
# Figure 5.1: Grouped within-subject AUC
# ------------------------------------------------------------

grouped_barplot(
    df=within_plot_df,
    x_col="dataset_label",
    y_col="within_subject_mean_auc",
    hue_col="model",
    output_path=FIGURES_DIR / "fig05_01_within_subject_auc.pdf",
    ylabel="Mean ROC-AUC",
    xlabel="Dataset",
    ylim=(0.5, 1.0),
    legend_title="Model",
)

In [ ]:
# ------------------------------------------------------------
# Figure 5.2: LOSO AUC
# ------------------------------------------------------------

grouped_barplot(
    df=loso_plot_df,
    x_col="dataset_label",
    y_col="loso_mean_auc",
    hue_col="model",
    output_path=FIGURES_DIR / "fig05_02_loso_auc.pdf",
    ylabel="Mean ROC-AUC",
    xlabel="Dataset",
    ylim=(0.5, 1.0),
    legend_title="Model",
)

In [ ]:
# ------------------------------------------------------------
# ERP waveform loading helpers
# ------------------------------------------------------------

def find_time_column(df):
    candidates = [
        "time",
        "time_s",
        "time_sec",
        "times",
        "latency",
        "latency_s",
        "t",
    ]

    lower_map = {c.lower(): c for c in df.columns}

    for c in candidates:
        if c in lower_map:
            return lower_map[c]

    return None


def find_dataset_column(df):
    candidates = ["dataset_label", "dataset", "dataset_id"]

    lower_map = {c.lower(): c for c in df.columns}

    for c in candidates:
        if c in lower_map:
            return lower_map[c]

    return None


def load_erp_waveform_table():
    """
    Try to find a saved ERP waveform/difference table.

    Accepted formats:
    1. Long format:
       dataset_label, time_s, condition, amplitude_uv
       where condition has target/standard or difference.

    2. Wide difference format:
       dataset_label, time_s, difference_uv

    3. Wide target/standard format:
       dataset_label, time_s, target_uv, standard_uv
    """

    candidates = find_all([
        "results/**/erp_sanity*/tables/*waveform*.csv",
        "results/**/erp_sanity*/tables/*erp*.csv",
        "results/**/erp*sanity*/**/*waveform*.csv",
        "results/**/erp*sanity*/**/*difference*.csv",
    ])

    print(f"Found {len(candidates)} ERP candidate CSV files.")

    for path in candidates:
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        if df.empty:
            continue

        time_col = find_time_column(df)
        dataset_col = find_dataset_column(df)

        if time_col is None:
            continue

        lower_cols = {c.lower(): c for c in df.columns}

        # Case 1: direct difference column
        diff_candidates = [
            "difference_uv",
            "diff_uv",
            "target_minus_standard_uv",
            "target_standard_diff_uv",
            "amplitude_diff_uv",
        ]

        diff_col = None
        for c in diff_candidates:
            if c in lower_cols:
                diff_col = lower_cols[c]
                break

        if diff_col is not None:
            out = df.copy()
            if dataset_col is None:
                out["dataset_label"] = path.stem
                dataset_col = "dataset_label"

            out = out[[dataset_col, time_col, diff_col]].copy()
            out.columns = ["dataset_label", "time_s", "difference_uv"]
            out["source_file"] = str(path)
            return out

        # Case 2: wide target/standard columns
        target_candidates = ["target_uv", "target_mean_uv", "target", "oddball_uv"]
        standard_candidates = ["standard_uv", "standard_mean_uv", "standard"]

        target_col = None
        standard_col = None

        for c in target_candidates:
            if c in lower_cols:
                target_col = lower_cols[c]
                break

        for c in standard_candidates:
            if c in lower_cols:
                standard_col = lower_cols[c]
                break

        if target_col is not None and standard_col is not None:
            out = df.copy()
            if dataset_col is None:
                out["dataset_label"] = path.stem
                dataset_col = "dataset_label"

            out = out[[dataset_col, time_col, target_col, standard_col]].copy()
            out.columns = ["dataset_label", "time_s", "target_uv", "standard_uv"]
            out["difference_uv"] = out["target_uv"] - out["standard_uv"]
            out["source_file"] = str(path)
            return out[["dataset_label", "time_s", "difference_uv", "source_file"]]

        # Case 3: long format with condition/amplitude
        condition_col = None
        amplitude_col = None

        for c in ["condition", "event_type", "class", "label"]:
            if c in lower_cols:
                condition_col = lower_cols[c]
                break

        for c in ["amplitude_uv", "mean_uv", "value_uv", "amplitude", "value"]:
            if c in lower_cols:
                amplitude_col = lower_cols[c]
                break

        if condition_col is not None and amplitude_col is not None:
            out = df.copy()
            if dataset_col is None:
                out["dataset_label"] = path.stem
                dataset_col = "dataset_label"

            out = out[[dataset_col, time_col, condition_col, amplitude_col]].copy()
            out.columns = ["dataset_label", "time_s", "condition", "amplitude_uv"]

            out["condition_clean"] = out["condition"].astype(str).str.lower()

            # Try target/standard pivot
            out["condition_group"] = np.nan
            out.loc[out["condition_clean"].str.contains("target|oddball"), "condition_group"] = "target"
            out.loc[out["condition_clean"].str.contains("standard|frequent"), "condition_group"] = "standard"
            out.loc[out["condition_clean"].str.contains("diff|difference"), "condition_group"] = "difference"

            if (out["condition_group"] == "difference").any():
                diff = out[out["condition_group"] == "difference"].copy()
                diff = diff[["dataset_label", "time_s", "amplitude_uv"]]
                diff = diff.rename(columns={"amplitude_uv": "difference_uv"})
                diff["source_file"] = str(path)
                return diff

            pivot = (
                out.dropna(subset=["condition_group"])
                .groupby(["dataset_label", "time_s", "condition_group"], as_index=False)["amplitude_uv"]
                .mean()
                .pivot_table(
                    index=["dataset_label", "time_s"],
                    columns="condition_group",
                    values="amplitude_uv",
                )
                .reset_index()
            )

            if {"target", "standard"}.issubset(pivot.columns):
                pivot["difference_uv"] = pivot["target"] - pivot["standard"]
                pivot["source_file"] = str(path)
                return pivot[["dataset_label", "time_s", "difference_uv", "source_file"]]

    return None

In [ ]:
# ------------------------------------------------------------
# Load ERP waveform result data if available
# ------------------------------------------------------------

erp_waveform_df = load_erp_waveform_table()

if erp_waveform_df is not None:
    print("Loaded ERP waveform table from:")
    print(erp_waveform_df["source_file"].iloc[0])
    display(erp_waveform_df.head())
else:
    print("No ERP waveform CSV table found.")
    print("Will assemble Figure 6.1 from saved ERP result figure artifacts.")

In [ ]:
# ------------------------------------------------------------
# Figure 6.1: ERP difference waveforms from data if available
# ------------------------------------------------------------

if erp_waveform_df is not None:
    erp_plot_df = erp_waveform_df.copy()

    erp_plot_df["dataset_label"] = erp_plot_df["dataset_label"].apply(clean_dataset_label)

    erp_plot_df = erp_plot_df.dropna(
        subset=["dataset_label", "time_s", "difference_uv"]
    ).copy()

    erp_plot_df["dataset_label"] = pd.Categorical(
        erp_plot_df["dataset_label"],
        categories=dataset_order,
        ordered=True,
    )

    erp_plot_df = erp_plot_df.sort_values(
        ["dataset_label", "time_s"]
    ).reset_index(drop=True)

    # Convert from volts to microvolts if necessary.
    erp_plot_df, erp_y_label, erp_scale_note = prepare_erp_difference_for_plot(
        erp_plot_df,
        value_col="difference_uv",
    )

    erp_plot_df.to_csv(
        TABLES_DIR / "fig06_01_erp_difference_waveforms_plot_data.csv",
        index=False,
    )

    fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
    axes = axes.ravel()

    for ax, dataset in zip(axes, dataset_order):
        sub = erp_plot_df[erp_plot_df["dataset_label"] == dataset].copy()

        if sub.empty:
            ax.set_title(dataset)
            ax.text(
                0.5,
                0.5,
                "No data",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.axis("off")
            continue

        ax.plot(
            sub["time_s"],
            sub["difference_plot"],
            linewidth=2.2,
        )

        ax.axhline(0, linewidth=1, linestyle="--", alpha=0.55)
        ax.axvline(0, linewidth=1, linestyle="--", alpha=0.55)

        ax.set_title(dataset)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel(erp_y_label)

        ax.grid(alpha=0.25)
        ax.tick_params(axis="both", labelsize=9)

        # Prevent scientific notation after conversion to microvolts.
        ax.ticklabel_format(axis="y", style="plain")

    fig.tight_layout()

    save_current_figure(
        FIGURES_DIR / "fig06_01_erp_difference_waveforms.pdf"
    )

    plt.show()

    print("Saved:", TABLES_DIR / "fig06_01_erp_difference_waveforms_plot_data.csv")

In [ ]:
# ------------------------------------------------------------
# Figure 6.1 fallback: assemble from saved ERP difference waveform images
# ------------------------------------------------------------

if erp_waveform_df is None:
    erp_image_patterns = {
        "ds003061": [
            "results/**/ds003061*max30_wide_timebin*difference_waveform.png",
            "results/**/ds003061*difference_waveform.png",
        ],
        "ds005863": [
            "results/**/ds005863*max30_wide_timebin*difference_waveform.png",
            "results/**/ds005863*difference_waveform.png",
        ],
        "ds006466": [
            "results/**/ds006466*max30_wide_timebin*difference_waveform.png",
            "results/**/ds006466*difference_waveform.png",
        ],
        "ds006480 pooled": [
            "results/**/ds006480*max30_wide_timebin*difference_waveform.png",
            "results/**/ds006480*difference_waveform.png",
        ],
    }

    erp_image_paths = {}
    for label, patterns in erp_image_patterns.items():
        path = find_first(patterns)
        erp_image_paths[label] = path
        print(label, ":", path)

    missing = [label for label, path in erp_image_paths.items() if path is None]
    if missing:
        raise FileNotFoundError(
            "Could not find ERP data table or saved ERP images for: "
            + ", ".join(missing)
        )

    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
    axes = axes.ravel()

    for ax, label in zip(axes, dataset_order):
        img = Image.open(erp_image_paths[label]).convert("RGB")
        ax.imshow(img)
        ax.set_title(label)
        ax.axis("off")
        copy_source_artifact(erp_image_paths[label])

    fig.tight_layout()

    save_current_figure(FIGURES_DIR / "fig06_01_erp_difference_waveforms.pdf")
    plt.show()

In [ ]:
# ------------------------------------------------------------
# Time-resolved decoding loading helper
# ------------------------------------------------------------

def load_time_resolved_table():
    """
    Try to find a saved time-resolved decoding table.

    Expected columns, flexible:
    - dataset_label or dataset
    - time_s / time / center_time_s / window_center_s
    - auc / mean_auc / roc_auc
    """

    candidates = find_all([
        "results/**/time_resolved*/tables/*.csv",
        "results/**/time*resolved*/**/*.csv",
        "results/**/time_resolved*auc*.csv",
    ])

    print(f"Found {len(candidates)} time-resolved candidate CSV files.")

    for path in candidates:
        try:
            df = pd.read_csv(path)
        except Exception:
            continue

        if df.empty:
            continue

        lower_cols = {c.lower(): c for c in df.columns}

        dataset_col = None
        for c in ["dataset_label", "dataset", "dataset_id"]:
            if c in lower_cols:
                dataset_col = lower_cols[c]
                break

        time_col = None
        for c in ["time_s", "time", "center_time_s", "window_center_s", "latency_s"]:
            if c in lower_cols:
                time_col = lower_cols[c]
                break

        auc_col = None
        for c in ["auc", "mean_auc", "roc_auc", "time_resolved_auc"]:
            if c in lower_cols:
                auc_col = lower_cols[c]
                break

        if dataset_col is not None and time_col is not None and auc_col is not None:
            out = df[[dataset_col, time_col, auc_col]].copy()
            out.columns = ["dataset_label", "time_s", "auc"]
            out["source_file"] = str(path)
            return out

    return None

In [ ]:
# ------------------------------------------------------------
# Figure 6.2: Time-resolved decoding
# ------------------------------------------------------------

time_resolved_df = load_time_resolved_table()

if time_resolved_df is not None:
    print("Loaded time-resolved decoding table from:")
    print(time_resolved_df["source_file"].iloc[0])

    time_plot_df = time_resolved_df.copy()
    time_plot_df["dataset_label"] = time_plot_df["dataset_label"].apply(clean_dataset_label)
    time_plot_df = time_plot_df.dropna(subset=["dataset_label", "time_s", "auc"]).copy()

    time_plot_df["dataset_label"] = pd.Categorical(
        time_plot_df["dataset_label"],
        categories=dataset_order,
        ordered=True,
    )

    time_plot_df = time_plot_df.sort_values(["dataset_label", "time_s"]).reset_index(drop=True)

    time_plot_df.to_csv(
        TABLES_DIR / "fig06_02_time_resolved_decoding_plot_data.csv",
        index=False,
    )

    fig, ax = plt.subplots(figsize=(10, 5.5))

    for dataset in dataset_order:
        sub = time_plot_df[time_plot_df["dataset_label"] == dataset]
        if sub.empty:
            continue

        ax.plot(
            sub["time_s"],
            sub["auc"],
            linewidth=2.2,
            label=dataset,
        )

    ax.axhline(0.5, linewidth=1, linestyle="--", alpha=0.5)
    ax.axvline(0, linewidth=1, linestyle="--", alpha=0.5)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("ROC-AUC")
    ax.set_ylim(0.45, max(0.85, time_plot_df["auc"].max() * 1.05))
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    fig.tight_layout()

    save_current_figure(FIGURES_DIR / "fig06_02_time_resolved_decoding.pdf")
    plt.show()

else:
    print("No time-resolved CSV table found.")
    print("Will convert saved time-resolved result image.")

    time_resolved_image_path = require_file(
        [
            "results/**/time_resolved_auc_all_datasets.png",
            "results/**/time*resolved*auc*all*datasets*.png",
        ],
        "time-resolved decoding result image",
    )

    convert_image_to_pdf(
        source_path=time_resolved_image_path,
        output_path=FIGURES_DIR / "fig06_02_time_resolved_decoding.pdf",
        width=11,
        height=6,
    )

In [ ]:
# ------------------------------------------------------------
# Load subject-vs-classifier spread summary
# ------------------------------------------------------------

spread_summary_path = require_file(
    [
        "results/**/overall_subject_vs_classifier_spread_zscored_summary.csv",
        "results/**/subject_vs_classifier_spread_summary.csv",
        "results/**/overall*spread*zscored*.csv",
    ],
    "subject-vs-classifier spread summary",
)

spread_df = pd.read_csv(spread_summary_path)
print("Loaded:", spread_summary_path)
display(spread_df.head())

In [ ]:
# ------------------------------------------------------------
# Prepare Figure 7.1 plotting data
# ------------------------------------------------------------

if {
    "evaluation_setting",
    "subject_auc_z_sd",
    "classifier_auc_z_sd",
}.issubset(spread_df.columns):

    variability_plot_df = spread_df[
        ["evaluation_setting", "subject_auc_z_sd", "classifier_auc_z_sd"]
    ].copy()

    variability_plot_df["evaluation_setting"] = variability_plot_df[
        "evaluation_setting"
    ].apply(normalize_evaluation_setting)

    variability_plot_df = variability_plot_df.melt(
        id_vars="evaluation_setting",
        value_vars=["subject_auc_z_sd", "classifier_auc_z_sd"],
        var_name="source_of_variability",
        value_name="auc_sd",
    )

    variability_plot_df["source_of_variability"] = variability_plot_df[
        "source_of_variability"
    ].replace({
        "subject_auc_z_sd": "Subject",
        "classifier_auc_z_sd": "Classifier",
    })

else:
    raise ValueError(
        "Could not find required z-scored variability columns.\n"
        "Required: evaluation_setting, subject_auc_z_sd, classifier_auc_z_sd\n"
        f"Available columns: {list(spread_df.columns)}"
    )

setting_order = ["Within-subject", "LOSO"]
source_order = ["Subject", "Classifier"]

variability_plot_df["evaluation_setting"] = pd.Categorical(
    variability_plot_df["evaluation_setting"],
    categories=setting_order,
    ordered=True,
)
variability_plot_df["source_of_variability"] = pd.Categorical(
    variability_plot_df["source_of_variability"],
    categories=source_order,
    ordered=True,
)

variability_plot_df = variability_plot_df.sort_values(
    ["evaluation_setting", "source_of_variability"]
).reset_index(drop=True)

variability_plot_df.to_csv(
    TABLES_DIR / "fig07_01_subject_vs_classifier_variability_plot_data.csv",
    index=False,
)

display(variability_plot_df)
print("Saved:", TABLES_DIR / "fig07_01_subject_vs_classifier_variability_plot_data.csv")

In [ ]:
# ------------------------------------------------------------
# Figure 7.1: Subject vs classifier variability
# ------------------------------------------------------------

grouped_barplot(
    df=variability_plot_df,
    x_col="evaluation_setting",
    y_col="auc_sd",
    hue_col="source_of_variability",
    output_path=FIGURES_DIR / "fig07_01_subject_vs_classifier_variability.pdf",
    ylabel="Standard deviation of dataset-z-scored AUC",
    xlabel="Evaluation setting",
    ylim=(0.0, max(1.05, variability_plot_df["auc_sd"].max() * 1.15)),
    legend_title="Variability source",
    width_inches=8,
    height_inches=5.2,
)

In [ ]:
# ------------------------------------------------------------
# Locate subject-quality-index/AUC table
# ------------------------------------------------------------

quality_auc_path = find_first([
    "results/**/auc_with_subject_quality_index.csv",
    "results/**/subject_quality_index_vs_auc_data.csv",
    "results/**/subject_quality*auc*.csv",
    "results/**/relative_importance*/tables/*quality*auc*.csv",
])

if quality_auc_path is None:
    raise FileNotFoundError(
        "Could not find a subject-quality-index/AUC table.\n"
        "Expected a table with columns like:\n"
        "  evaluation_setting, dataset_label, model, auc, subject_quality_index\n"
        "Search patterns tried:\n"
        "  results/**/auc_with_subject_quality_index.csv\n"
        "  results/**/subject_quality_index_vs_auc_data.csv\n"
        "  results/**/subject_quality*auc*.csv\n"
        "  results/**/relative_importance*/tables/*quality*auc*.csv\n"
        "\n"
        "If your existing figure exists but the table was not saved, re-run the relative-importance notebook "
        "with an added export of the plotting dataframe."
    )

quality_auc_df = pd.read_csv(quality_auc_path)
print("Loaded:", quality_auc_path)
display(quality_auc_df.head())

In [ ]:
# ------------------------------------------------------------
# Normalize subject-quality/AUC columns
# ------------------------------------------------------------

df = quality_auc_df.copy()
lower_cols = {c.lower(): c for c in df.columns}

# Flexible column detection
evaluation_col = None
for c in ["evaluation_setting", "setting", "eval_setting"]:
    if c in lower_cols:
        evaluation_col = lower_cols[c]
        break

dataset_col = None
for c in ["dataset_label", "dataset", "dataset_id"]:
    if c in lower_cols:
        dataset_col = lower_cols[c]
        break

model_col = None
for c in ["model", "classifier"]:
    if c in lower_cols:
        model_col = lower_cols[c]
        break

auc_col = None
for c in ["auc", "mean_auc", "subject_auc", "roc_auc", "auc_z"]:
    if c in lower_cols:
        auc_col = lower_cols[c]
        break

quality_col = None
for c in ["subject_quality_index", "quality_index", "subject_quality", "signal_quality_index"]:
    if c in lower_cols:
        quality_col = lower_cols[c]
        break

detected = {
    "evaluation_col": evaluation_col,
    "dataset_col": dataset_col,
    "model_col": model_col,
    "auc_col": auc_col,
    "quality_col": quality_col,
}

print("Detected columns:", detected)

missing_detected = [k for k, v in detected.items() if v is None]
if missing_detected:
    raise ValueError(
        "Could not detect required columns for subject-quality plot.\n"
        f"Missing: {missing_detected}\n"
        f"Available columns: {list(df.columns)}"
    )

quality_plot_df = df[
    [evaluation_col, dataset_col, model_col, auc_col, quality_col]
].copy()

quality_plot_df.columns = [
    "evaluation_setting",
    "dataset_label",
    "model",
    "auc",
    "subject_quality_index",
]

quality_plot_df["evaluation_setting"] = quality_plot_df[
    "evaluation_setting"
].apply(normalize_evaluation_setting)

quality_plot_df["dataset_label"] = quality_plot_df[
    "dataset_label"
].apply(clean_dataset_label)

quality_plot_df["model"] = quality_plot_df["model"].apply(clean_model_label)

quality_plot_df = quality_plot_df.dropna(
    subset=["evaluation_setting", "dataset_label", "model", "auc", "subject_quality_index"]
).copy()

# If AUC is not already z-scored, create dataset-z-scored AUC for pooled plotting.
if auc_col.lower() == "auc_z":
    quality_plot_df["auc_z"] = quality_plot_df["auc"]
else:
    quality_plot_df["auc_z"] = quality_plot_df.groupby("dataset_label")["auc"].transform(
        lambda s: (s - s.mean()) / s.std(ddof=0)
    )

quality_plot_df = quality_plot_df.dropna(subset=["auc_z"]).copy()

quality_plot_df.to_csv(
    TABLES_DIR / "fig07_02_subject_quality_vs_auc_plot_data.csv",
    index=False,
)

display(quality_plot_df.head())
print("Saved:", TABLES_DIR / "fig07_02_subject_quality_vs_auc_plot_data.csv")

In [ ]:
# ------------------------------------------------------------
# Figure 7.2: Subject-quality index vs AUC
# ------------------------------------------------------------

setting_order = ["Within-subject", "LOSO"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, setting in zip(axes, setting_order):
    sub = quality_plot_df[
        quality_plot_df["evaluation_setting"] == setting
        ].copy()

    ax.scatter(
        sub["subject_quality_index"],
        sub["auc_z"],
        alpha=0.65,
        s=28,
        edgecolor="none",
    )

    # Linear trend from saved data
    if len(sub) >= 3:
        x = sub["subject_quality_index"].to_numpy()
        y = sub["auc_z"].to_numpy()

        finite = np.isfinite(x) & np.isfinite(y)
        x = x[finite]
        y = y[finite]

        if len(x) >= 3 and np.std(x) > 0:
            slope, intercept = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 100)
            y_line = slope * x_line + intercept
            ax.plot(x_line, y_line, linewidth=2)

    ax.axhline(0, linewidth=1, linestyle="--", alpha=0.5)
    ax.axvline(0, linewidth=1, linestyle="--", alpha=0.5)
    ax.set_title(setting)
    ax.set_xlabel("Subject-quality index")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Dataset-z-scored AUC")

fig.tight_layout()

save_current_figure(FIGURES_DIR / "fig07_02_subject_quality_vs_auc.pdf")
plt.show()

In [ ]:
# ------------------------------------------------------------
# Load incremental R² table
# ------------------------------------------------------------

incremental_r2_path = require_file(
    [
        "results/**/incremental_r2_subject_quality_vs_classifier.csv",
        "results/**/incremental*r2*subject*quality*classifier*.csv",
    ],
    "incremental R2 subject-quality vs classifier table",
)

incremental_r2_df_raw = pd.read_csv(incremental_r2_path)
print("Loaded:", incremental_r2_path)

display(incremental_r2_df_raw.head(20))

In [ ]:
# ------------------------------------------------------------
# Prepare Figure 7.3 plotting data
# ------------------------------------------------------------

require_columns(
    incremental_r2_df_raw,
    ["evaluation_setting", "comparison", "r2_increment"],
    "incremental R2 table",
)

wanted_comparisons = {
    "classifier_added_after_dataset": "Classifier identity",
    "subject_quality_added_after_dataset": "Subject-quality index",
}

incremental_r2_plot_df = incremental_r2_df_raw[
    incremental_r2_df_raw["comparison"].isin(wanted_comparisons.keys())
].copy()

if incremental_r2_plot_df.empty:
    raise ValueError(
        "Incremental R2 table does not contain the required comparisons:\n"
        f"{list(wanted_comparisons.keys())}\n"
        f"Available comparisons: {incremental_r2_df_raw['comparison'].unique()}"
    )

incremental_r2_plot_df["evaluation_setting"] = incremental_r2_plot_df[
    "evaluation_setting"
].apply(normalize_evaluation_setting)

incremental_r2_plot_df["predictor_added_after_dataset"] = incremental_r2_plot_df[
    "comparison"
].map(wanted_comparisons)

incremental_r2_plot_df = incremental_r2_plot_df[
    [
        "evaluation_setting",
        "predictor_added_after_dataset",
        "r2_increment",
    ]
].rename(columns={"r2_increment": "incremental_r2"})

setting_order = ["Within-subject", "LOSO"]
predictor_order = ["Classifier identity", "Subject-quality index"]

incremental_r2_plot_df["evaluation_setting"] = pd.Categorical(
    incremental_r2_plot_df["evaluation_setting"],
    categories=setting_order,
    ordered=True,
)

incremental_r2_plot_df["predictor_added_after_dataset"] = pd.Categorical(
    incremental_r2_plot_df["predictor_added_after_dataset"],
    categories=predictor_order,
    ordered=True,
)

incremental_r2_plot_df = incremental_r2_plot_df.sort_values(
    ["evaluation_setting", "predictor_added_after_dataset"]
).reset_index(drop=True)

incremental_r2_plot_df.to_csv(
    TABLES_DIR / "fig07_03_incremental_r2_plot_data.csv",
    index=False,
)

display(incremental_r2_plot_df)
print("Saved:", TABLES_DIR / "fig07_03_incremental_r2_plot_data.csv")

In [ ]:
# ------------------------------------------------------------
# Figure 7.3: Incremental R²
# ------------------------------------------------------------

grouped_barplot(
    df=incremental_r2_plot_df,
    x_col="evaluation_setting",
    y_col="incremental_r2",
    hue_col="predictor_added_after_dataset",
    output_path=FIGURES_DIR / "fig07_03_incremental_r2.pdf",
    ylabel="Incremental $R^2$ after accounting for dataset",
    xlabel="Evaluation setting",
    ylim=(0.0, max(0.14, incremental_r2_plot_df["incremental_r2"].max() * 1.15)),
    legend_title="Predictor",
    width_inches=8,
    height_inches=5.2,
)

In [ ]:
# ------------------------------------------------------------
# Save final artifact manifest
# ------------------------------------------------------------

manifest_rows = []

for path in sorted(FIGURES_DIR.glob("*")):
    manifest_rows.append({
        "artifact_type": "figure",
        "filename": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_kb": round(path.stat().st_size / 1024, 1),
    })

for path in sorted(TABLES_DIR.glob("*")):
    manifest_rows.append({
        "artifact_type": "table",
        "filename": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_kb": round(path.stat().st_size / 1024, 1),
    })

for path in sorted(SOURCE_COPY_DIR.glob("*")):
    manifest_rows.append({
        "artifact_type": "source_artifact",
        "filename": path.name,
        "relative_path": str(path.relative_to(PROJECT_ROOT)),
        "size_kb": round(path.stat().st_size / 1024, 1),
    })

manifest_df = pd.DataFrame(manifest_rows)

manifest_path = ARTIFACT_DIR / "dissertation_final_figures_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

display(manifest_df)
print("Saved manifest:", manifest_path)

In [ ]:
# ------------------------------------------------------------
# Verify final dissertation figure outputs
# ------------------------------------------------------------

expected_outputs = [
    "fig05_01_within_subject_auc.pdf",
    "fig05_02_loso_auc.pdf",
    "fig06_01_erp_difference_waveforms.pdf",
    "fig06_02_time_resolved_decoding.pdf",
    "fig07_01_subject_vs_classifier_variability.pdf",
    "fig07_02_subject_quality_vs_auc.pdf",
    "fig07_03_incremental_r2.pdf",
]

summary = []
for filename in expected_outputs:
    path = FIGURES_DIR / filename
    summary.append({
        "figure": filename,
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
        "path": str(path),
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

missing = summary_df.loc[~summary_df["exists"], "figure"].tolist()
if missing:
    raise FileNotFoundError("Missing final figure outputs: " + ", ".join(missing))

print("All final dissertation figures were created successfully under:")
print(FIGURES_DIR)

# LaTeX inclusion commands

Use the following LaTeX commands in the dissertation.

In [ ]:
latex_commands = r"""
% Chapter 5 — Within-subject benchmark
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{results/thesis_artifacts/dissertation_final_figures/figures/fig05_01_within_subject_auc.pdf}
\caption{Grouped within-subject ROC-AUC across datasets and models. Linear models were strongest or nearly strongest in the within-subject setting.}
\label{fig:within_subject_auc}
\end{figure}

% Chapter 5 — LOSO benchmark
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{results/thesis_artifacts/dissertation_final_figures/figures/fig05_02_loso_auc.pdf}
\caption{LOSO ROC-AUC across datasets and models. EEGNet achieved the strongest LOSO performance across all four datasets.}
\label{fig:loso_auc}
\end{figure}

% Chapter 6 — ERP difference waveforms
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{results/thesis_artifacts/dissertation_final_figures/figures/fig06_01_erp_difference_waveforms.pdf}
\caption{Target-standard difference waveforms across datasets. The plots show task-related post-stimulus differences in plausible ERP-related windows, but they should not be interpreted as identical textbook P300 morphology across datasets.}
\label{fig:erp_sanity}
\end{figure}

% Chapter 6 — Time-resolved decoding
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{results/thesis_artifacts/dissertation_final_figures/figures/fig06_02_time_resolved_decoding.pdf}
\caption{Time-resolved decoding across datasets. Peak decoding occurred in post-stimulus windows consistent with ERP-related target-standard discrimination.}
\label{fig:time_resolved_decoding}
\end{figure}

% Chapter 7 — Subject vs classifier variability
\begin{figure}[htbp]
\centering
\includegraphics[width=0.85\textwidth]{results/thesis_artifacts/dissertation_final_figures/figures/fig07_01_subject_vs_classifier_variability.pdf}
\caption{Subject-level versus classifier-level variability in dataset-z-scored AUC. Subject-level variability was approximately 2.8 times larger than classifier-level variability in both within-subject and LOSO settings.}
\label{fig:subject_vs_classifier_variability}
\end{figure}

% Chapter 7 — Subject-quality index vs AUC
\begin{figure}[htbp]
\centering
\includegraphics[width=0.85\textwidth]{results/thesis_artifacts/dissertation_final_figures/figures/fig07_02_subject_quality_vs_auc.pdf}
\caption{Relationship between subject-quality index and decoding performance. Both variables are z-scored within dataset. Higher subject-quality values indicate stronger ERP/P300 separability and lower noise, and are associated with higher classification performance.}
\label{fig:subject_quality_vs_auc}
\end{figure}

% Chapter 7 — Incremental R2
\begin{figure}[htbp]
\centering
\includegraphics[width=\textwidth]{results/thesis_artifacts/dissertation_final_figures/figures/fig07_03_incremental_r2.pdf}
\caption{Incremental \(R^2\) comparison between classifier identity and subject-quality index. After accounting for dataset, subject-quality features explained more additional AUC variance than classifier identity in both within-subject and LOSO settings.}
\label{fig:incremental_r2}
\end{figure}
"""

print(latex_commands)